# VLM 파싱 모델 QWEN3 사용하기



## 1. 구글 드라이버 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

FILE_DIR = "/content/drive/MyDrive/3team_project"
OUTPUT_DIR = "/content/drive/MyDrive/3team_project/vlm_parsed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT 경로 폴더 생성 : {OUTPUT_DIR}")

OUTPUT 경로 폴더 생성 : /content/drive/MyDrive/3team_project/vlm_parsed


## 2. 패키지 설치

In [3]:
!pip install -q pymupdf Pillow tqdm pandas numpy
!pip install -q openai
!pip install -q transformers>=4.51.0 accelerate qwen-vl-utils
!pip install -q bitsandbytes # INT4 양자화용
!pip install -q langchain langchain-experimental langchain-openai
!pip install -q rouge-score jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10

각 패키지의 용도는 다음과 같습니다:

- `pymupdf`, `Pillow`: 문서를 이미지로 변환하고 처리합니다.
(HWP 파일은 보통 PDF로 변환 후, 이 라이브러리들을 통해 이미지로 만들어 AI에게 보여줍니다.)
- `openai`: OpenAI의 GPT-4o(Vision) 모델을 사용하기 위한 라이브러리입니다.
- `transformers`, `qwen-vl-utils`: Qwen2-VL (Qwen3라고 지칭하신 최신 모델 등)을 로컬에서 불러오고 처리하는 데 필요합니다.
- `accelerate`, `bitsandbytes`: Colab GPU 메모리 한계 내에서 거대 모델(Qwen)을 효율적으로 돌리기 위한 최적화(양자화) 도구입니다.
- `langchain` 관련: AI 모델과 데이터를 쉽게 연결해주는 프레임워크입니다.
- `rouge-score`, `jiwer`: 파싱 결과가 얼마나 정확한지 평가(채점)하는 도구입니다.

참고: HWP 파일을 직접 읽는 도구는 포함되어 있지 않습니다.
보통 HWP → PDF 변환 후, PDF를 pymupdf로 이미지화하여 Qwen/OpenAI에게 시각적으로 읽게 하는 방식을 사용합니다.

In [15]:
# LibreOffice 설치 (HWP -> PDF 변환용, Colab 환경에서 필수)
!apt-get install -y libreoffice > /dev/null 2>&1
print("LibreOffice 설치 완료")

LibreOffice 설치 완료


## 3. 모델 로드

허깅페이스에서 Qwen3-VL-8B 모델을 로드합니다.

In [4]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# 모델 불러오기
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
   MODEL_ID , dtype="auto", device_map ="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f'모델 로드 완료 : {MODEL_ID}')

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

모델 로드 완료 : Qwen/Qwen3-VL-8B-Instruct


# 4. 유틸리시 함수
`norm` : 파일 이름 매칭

`convert_hwp_to_pdf` : hwp -> pdf 로 변환

`extract_base_md` : pdf -> md 로 변환

`safe_json_loads`  : 코드 블록 및 제어 문자 제거

In [5]:
import unicodedata

def norm(s: str) -> str:
  """NFC 정규화 (파일명 매칭용)"""
  return unicodedata.normalize("NFC", str(s).strip())

### 4-1. `norm` 함수가 필요한 이유 (NFC 정규화)

이 함수는 **한글 파일명 깨짐이나 매칭 실패**를 방지하기 위해 필수적입니다. 이유는 다음과 같습니다:

1.  **운영체제 간의 차이 (NFC vs NFD)**
    *   **Windows/Linux**: 한글을 '완성형(NFC)'으로 저장합니다. (예: `가`)
    *   **macOS**: 한글을 '조합형(NFD)'으로 풀어헤쳐서 저장하는 경우가 많습니다. (예: `ㄱ` + `ㅏ`)

2.  **데이터 불일치 문제**
    *   사람 눈에는 `파일.hwp`로 똑같이 보이지만, 컴퓨터 내부적으로는 서로 다른 문자열로 인식될 수 있습니다.
    *   예: `"한글" == "ㅎ+ㅏ+ㄴ+ㄱ+ㅡ+ㄹ"` 은 프로그래밍 언어에서 `False`가 됩니다.

3.  **해결책**
    *   `unicodedata.normalize("NFC", s)`를 사용하면, 운영체제 상관없이 모든 문자열을 **강제로 완성형(NFC)**으로 통일시킵니다.
    *   구글 드라이브는 여러 OS에서 파일이 업로드될 수 있으므로, 파일명을 비교하거나 찾을 때 이 함수를 통과시켜야 안전하게 처리할 수 있습니다.

In [6]:
import shutil

def free_gb(path="."):
  """디스크 잔여 용량 체크"""
  return shutil.disk_usage(path).free / (1024**3)

In [7]:
import json
import re

def safe_json_loads(s: str):
  """
  LLM 출력에서 JSON 부분만 추출하여 파싱합니다.
  실패 시 프로그램이 멈추지 않도록 빈 테이블 리스트를 반환합니다.
  """
  try:
    s = s.strip()
    # 1. 마크다운 코드 블록 (```json ... ```) 내부의 내용만 추출
    match = re.search(r"```(json)?(.*?)```", s, re.DOTALL | re.IGNORECASE)
    if match:
      s = match.group(2).strip()
    # 2. JSON 파싱 시도
    return json.loads(s)
  except:
    pass # 1차 시도 실패 시 아래로 진행

  # 3. 2차 시도: 가장 바깥쪽 { } 또는 [ ] 를 찾아 파싱
  try:
    match = re.search(r"(\{.*\}|\[.*\])", s, re.DOTALL)
    if match:
      return json.loads(match.group(0))
  except:
    pass

  # 4. 모든 파싱 실패 시: 빈 결과 반환 (None 반환 방지)
  return {"tables": []}

### 4-2 `safe_json_loads` 동작원리
  1. s.strip(): 문자열 앞뒤의 불필요한 공백/줄바꿈 제거
  2. re.search: 정규표현식을 이용해 ```json ... ``` 블록 내부 추출
     - re.DOTALL: 줄바꿈 문자를 포함하여 검색 (JSON은 여러 줄인 경우가 많음)
     - re.IGNORECASE: json/JSON 대소문자 구분 없이 매칭
  3. match.group(2): 패턴 내의 두 번째 그룹 (JSON 알맹이) 추출
  4. json.loads: 문자열을 파이썬 딕셔너리로 변환
  5. 예외 처리: 구조가 깨진 경우 가장 바깥쪽 { } 또는 [ ] 를 찾아 재시도

In [8]:
def insert_tables_to_content(md_text: str, tables_by_page: list):
  """기본 MD 텍스트에 VLM 추출 표 삽입"""
  lines = md_text.splitlines()
  out = []
  page_to_tables = {rec["page"]: rec["tables"] for rec in tables_by_page}
  current_page = None

  for line in lines:
    out.append(line)
    m = re.match(r"<!-- page: (\d+) -->", line)
    if m:
      current_page = int(m.group(1))
      for t_idx , t in enumerate(page_to_tables[current_page],1):
        cap = (t.get("caption") or "").strip() or f"표 {t_idx}"
        out.append(f"\n### [표] {cap}")
        out.append((t.get("markdown") or "").strip() + "\n")
  return "\n".join(out)

### 4-3 `insert_tables_to_content` 동작원리
  1. m = re.match(...)와 m.group(1)의 정체
  re.match는 특정 문자열이 패턴으로 시작하는지 확인하는 함수입니다.

   * 패턴 분석: `r"<!-- page: (\d+) -->"`
       * `<!-- page:`  : 이 문자열로 시작해야 함.
       * `(\d+)` : 캡처 그룹(Capturing Group)입니다. \d+는 1개 이상의 숫자(0-9)를 의미하며, 소괄호 ()로 감싸면 "이 부분만
         따로 추출하겠다"는 뜻입니다.
       *  `-->` : 마지막에 이 문자열이 와야 함.
   * `m.group(1)`의 역할:
       * m은 매칭 결과가 담긴 Match 객체입니다.
       * group(0)은 매치된 전체 문자열(<!-- page: 1 -->)을 반환합니다.
       * group(1)은 소괄호 () 안에 들어있는 첫 번째 그룹, 즉 페이지 번호 숫자(1)만 쏙 뽑아냅니다.

  2. 문자는 어디서 읽어가는가?
 `for line in lines:` 루프가 돌면서 매 줄(line)마다 `re.match`가
  실시간으로 검사합니다.

```python
 for line in lines: # 문서의 모든 줄을 하나씩 읽으면서
  out.append(line)
  m = re.match(...) # 현재 읽고 있는 이 줄이 "<!-- page: n -->" 형태인지 확인!
  if m: # 맞다면 (매칭 성공 시)
    current_page = int(m.group(1)) # 그 줄에서 숫자만 추출해서 현재 페이지 번호로 저장
```

  3. rec["page"] (Dictionary Comprehension)
  rec 은 데이터를 조회하기 쉽게 변환하는 과정입니다.


   * 원본 데이터 (`tables_by_page`): [{"page": 1, "tables": [...]}, {"page": 2, "tables": [...]}] 형태의 리스트입니다.
   * 변환 코드: page_to_tables = {rec["page"]: rec["tables"] for rec in tables_by_page}
       * rec는 "record"의 약자로 관습적으로 쓴 변수명입니다.
       * 리스트를 돌면서 1: [표데이터], 2: [표데이터] 같은 딕셔너리(Map) 형태로 바꿉니다.
       * 이렇게 해야 나중에 if current_page in page_to_tables: 처럼 특정 페이지의 표가 있는지 매우 빠르게(O(1)) 찾을 수
         있습니다.


  요약하자면:
   1. re.match가 매 줄마다 "페이지 구분자"인지 감시합니다.
   2. 구분자를 만나면 group(1)로 숫자만 따옵니다.
   3. 그 숫자를 키(Key)로 삼아 page_to_tables 딕셔너리에서 해당 페이지에 삽입할 표가 있는지 조회하여 끼워넣는
      방식입니다.

  rec["page"]나 rec["tables"] 값이란?

  이 값들은 함수 내부에서 정의된 것이 아니라, 함수를 호출하기 전 메인 파이프라인에서 데이터를 담을
  때 약속한 이름(Key)입니다.

  이름이 정의된 흐름을 단계별로 추적해 보겠습니다.

  ---


  1. 데이터의 탄생 (메인 루프의 Step 2)
  코드 하단의 for pdf_path in tqdm(pdfs, ...): 루프를 보시면 데이터를 생성하는 부분이 있습니다.

```python
# 1. Qwen 모델이 표를 뽑아냅니다 (result는 딕셔너리 형태)
result = extract_tables_with_qwen(pil_img, page_no, model, processor)

# 2. 뽑아낸 데이터를 'all_extracted_tables'라는 리스트에 넣습니다.
# 여기서 "page"와 "tables"라는 이름을 처음 정의하여 딕셔너리를 만듭니다!
if result.get("tables"):
  all_extracted_tables.append({
    "page": page_no,      # "page"라는 이름으로 페이지 번호를 저장
    "tables": result["tables"]  # "tables"라는 이름으로 표 데이터를 저장
    })
```
   * 여기서 `{"page": 1, "tables": [...]}` 같은 모양의 덩어리(Record)가 만들어집니다.

  ---

  2. 함수로 데이터 전달
  위에서 만든 all_extracted_tables 리스트가 insert_tables_to_content 함수의 두 번째 인자인 tables_by_page로 전달됩니다.
```python
# 함수 호출 부분
final_md = insert_tables_to_content(base_md_content, all_extracted_tables)
```
  ---


  3. rec는 무엇인가? (Dictionary Comprehension)
  함수 내부의 이 한 줄이 가장 헷갈리실 부분입니다.
  page_to_tables = {rec["page"]: rec["tables"] for rec in tables_by_page}


   * `rec` (Record의 약자): 이 변수는 파이썬의 리스트 컴프리헨션(또는 딕셔너리 컴프리헨션) 문법에서 쓰이는 임시 반복
     변수입니다. for i in range(10):에서 i와 같은 역할입니다.
   * 작동 원리: tables_by_page라는 리스트 안에 들어있는 덩어리들을 하나씩 꺼내서 rec라고 부르기로 합니다.
   * 조회: rec는 아까 1번 단계에서 만든 {"page": 1, "tables": [...]} 형태의 딕셔너리이므로, rec["page"]라고 하면 페이지
     번호(1)가 나오고, rec["tables"]라고 하면 표 리스트가 나옵니다.

  ---

  요약하자면
   1. `rec`: 반복문(for) 안에서만 쓰이는 임시 이름입니다.
   2. `["page"]`, `["tables"]`: 메인 파이프라인에서 데이터를 만들 때 사용한 Key 이름입니다.
   3. 흐름: 메인 코드에서 이름 지어줌 -> 함수에 전달 -> 함수 내 반복문에서 그 이름으로 꺼내 씀 구조입니다.

In [9]:
def extract_table_with_qwen(pil_image, page_no, model, processor):
    """Qwen3-VL 전용 추론 로직"""
    SYSTEM_PROMPT = """너는 문서 페이지에서 '표'만 추출하는 도우미다.
    보이는 표를 Markdown table로 변환하라. 설명 없이 JSON으로만 답하라.
    출력 형식 : {"tables":[{"caption": "제목", "markdown": "|...|"}]}"""
    messages = [
        {"role":"system", "content":[{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image":pil_image},
                {"type": "text", "text": f"{page_no}페이지에서 표를 찾아 JSON으로 추출해줘."}
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
    return safe_json_loads(output_text)

`apply_chat_template` 메서드 파라미터(매개변수):
- `conversation` : 포맷(형식)을 맞출 대화 내용입니다.
- `chat_template` (선택사항) : 대화의 포맷을 맞추는 데 사용할 Jinja 템플릿 입니다. 값을 주지 않으면 토크나이저에 내장된 기본 채팅 템플릿이 사용됩니다.
- 설명:<br/>
토크나이저의 `apply_chat_template` 메서드와 마찬가지로, 이 메서드는 입력된 대화(conversation)에 Jinja 템플릿을 적용하여 **'토큰화(tokenize) 할 수 있는 단일 문자열로 변환** 합니다.<br/>
입력값은 문서 하단의 예시와 같은 형식이어야 하며, 각 메시지의 `content`는 텍스트와 (선택적으로) 이미지 또는 비디오 입력으로 구성된 리스트 형태여야 합니다.

messages 리스트 안에 `role`(system, user)과 `content`(이미지, 텍스트)를 딕셔너리 형태로 정리하였습니다. QWEN 모델은 파이썬 리스트와 딕셔너리 구조를 직접 읽을 수 없습니다. 모델이 학습할 때 사용했던 자신만의 특정한 '문자열 대화 규칙'이 있습니다.

그래서 `processor.apply_chat_template`이 파이썬 딕셔너리를 모델 전용 문자로 바꿔주는 번역기 역할을 합니다.

`processor.apply_chat_template` 메서드의 파라미터
- `messages` : 우리가 파이썬 문법으로 작성한 대화 리스트를 넘겨줍니다.
- `tokenize=False` : 텍스트를 토큰으로 변환하지 말고 문자열 상태로 변환하라는 뜻입니다.
- `add_generation_prompt=True` : 문자열  맨 마지막에 "이제 AI(어시스턴스)가 답변할 차례야" 라는 프롬프트를 자동으로 덧붙여 달라는 뜻입니다.

`image_inputs, _ = process_vision_info(messages)` 코드 해석

- `process_vision_info(message)` : 입력된 메시지 목록에서 이미지가 있는지, 비디오가 있는지 스캔하고 데이터를 가져옵니다.

- **반환값의 비밀** : 이 함수는 항상 두 개의 값을 세트로 반환합니다. 첫 번째는 **이미지 데이터 리스트**이고, 두 번째는 **비디오 데이터 리스트** 입니다.

- `_`(언더 스코어) : 파이썬에서 `_`는 "이 반환값은 쓸 데 없으니 변수에 저장하지 않고 버리겠다"는 의미의 관례적인 기호입니다. 지금은 문서(이미지)에서 표를 추출하는 작업 중이므로 비디오 데이터는 필요가 없습니다. 따라서 비디오를 받는 두 번째 자리를 빈칸처럼 `_`로 처리한 것입니다.

- `image_inputs` : 결과적으로 추출된 순수한 이미지의 데이터들만 이 변수에 담기에 됩니다.

1. `torch.no_grad()`: 지금 학습(Training)할 때가 아니니, 미분 계산(Gradient)을 하지 말고 메모리를 아껴라는 뜻입니다.

2. `Trimming` : 출력된 토큰 다듬기

**AI가 뱉어낸 답변에서, 내가 처음에 했던 질문(프롬프트)은 잘라내고 순수하게 새로 만들어진 답변만 남겨줘**
- 왜 자르는가? : 대부분 텍스트 생성 AI 모델(model.generate)은 정답을 내놓을 때 **[내가 입력한 질문 + AI가 생성한 답변]**을 하나로 이어서 반환하는 특징이 있습니다.
- 동작원리 : `len(in_ids)` : 내가 입력한 질문의 길이(토큰 갯수)를 잽니다.
    - `out_ids[len(in_ids):]` : 전체 출력 결과(`out_ids`)에서 딱 내 질문 길이만큼 앞부분을 자르고 뒤부터 끝까지 글만 가져옵니다.

3. `output_text` : 숫자를 사람이 읽을 수 있는 글자로 변환하기
임베딩값을 문자로 변경해주는 역할을 합니다.
```python
output_text = processor.batch_decode(generated_ids_trimmed, skip_speical_tokens=True)[0]
```
- `batch_decode` : 임베딩 값을 문자로 변환해주는 함수
- `skip_special_tokens=True` : AI가 내부적으로 사용하는 특수 기호들을 화면에서 보이지 않게 깔끔하게 지워줍니다. (ex. `<|im_start|>`, `<|endoftext|>`)
- `[0]` : 결과를 리스트에서 꺼내어 순수한 문자열 형태로 변환합니다. (왜 꼭 [0] 해야하는지 궁금)

4. `return safe_json_loads(output_text)` 의 반환값
python 에서 `return 함수()` 형태로 작성되어 있다면, 그 함수를 먼저 실행 한 뒤, 그 함수가 내놓은 결과물을 이 함수의 최종 반환값으로 던져주겠다는 의미입니다.

`safe_json_loads` 함수의 짧은 정리
    - 1. AI가 생성한 마크다운 텍스트(`output_text`)를 `safe_json_load` 함수에 집어 넣습니다.
    - 2. `safe_json_load` 함수는 정규 표현식을 써서 텍스트 안에 있는 JSON만 뽑아서 파이썬 딕셔너리(Dictionary) 객체로 변환합니다.
    - 최종 반환값 : `extract_table_with_qwen` 함수를 실행하고 나면, 텍스트가 아니라 파이썬에서 바로 다룰 수 있는 깔끔한 딕셔너리 데이터가 나옵니다.


In [25]:
import subprocess

def convert_hwp_to_pdf(source_dir, output_dir):
  """
  지정된 폴더(source_dir) 내의 모든 .hwp 파일을 찾아 PDF로 변환합니다.
  """
  os.makedirs(output_dir, exist_ok=True)

  # 소스 디렉토리 내의 모든 파일을 확인
  if not os.path.exists(source_dir):
    print(f"경로를 찾을 수 없습니다 : {source_dir}")
    return

  all_files = os.listdir(source_dir)

  # .hwp 확장자만 가진 파일만 골라냅니다.
  hwp_files =[ f for f in all_files if f.lower().endswith('.hwp')]
  print(f"발견된 HWP 파일 갯수 : {len(hwp_files)}")
  sucess_count = 0

  # 각 HWP 파일에 대해 변환 명령을 실행합니다.
  for file_name in hwp_files:
    input_path = os.path.join(source_dir, file_name)
    stem = os.path.splitext(file_name)[0]
    odt_path = os.path.join(output_dir, stem + ".odt")
    pdf_path = os.path.join(output_dir, stem + ".pdf")
    print(f"변환 중 : {file_name} ... ")

    try:
            # Step 1: HWP → ODT (pyhwp)
            r1 = subprocess.run(
                ["hwp5odt", input_path],
                capture_output=True, text=True, cwd=output_dir
            )
            print(f"  [hwp5odt] returncode={r1.returncode}")
            if r1.stderr: print(f"  [hwp5odt] stderr: {r1.stderr[:200]}")

            if not os.path.exists(odt_path):
                print(f" HWP→ODT 실패 (odt 파일 없음)")
                continue

            # Step 2: ODT → PDF (LibreOffice)
            r2 = subprocess.run(
                ["libreoffice", "--headless", "--convert-to", "pdf",
                 odt_path, "--outdir", output_dir],
                capture_output=True, text=True
            )
            print(f"  [libreoffice] returncode={r2.returncode}")
            if r2.stderr: print(f"  [libreoffice] stderr: {r2.stderr[:200]}")

            if os.path.exists(pdf_path):
                print(f"변환 완료: {pdf_path}")
                success_count += 1
            else:
                print(f"ODT→PDF 실패 (pdf 파일 없음)")


    except FileNotFoundError as e:
      print(f"명령어 없음: {e}")
    except subprocess.CalledProcessError as e:
      print(f"변환 실패 : {e.stderr}")
    except Exception as e:
            print(f"오류: {e}")


  print(f"작업 완료 : (성공 : {sucess_count}/{len(hwp_files)})")

### 4-1 LibreOffice 변환 명령어 구성:
1. "libreoffice": 리브레오피스 프로그램 실행
2. "--headless": GUI 없이 백그라운드에서 실행 (서버/Colab 환경 필수)
3. "--convert-to", "pdf": 입력 파일을 PDF로 변환
4. FILE_DIR: 변환할 대상 파일의 경로
5. "--outdir", OUTPUT_DIR: 변환된 결과물이 저장될 폴더 지정

### 4-2. 외부 프로세스 제어 (subprocess)

VLM 파싱 과정에서 외부 도구(LibreOffice 등)를 실행하기 위해 `subprocess` 모듈을 사용합니다.

### 핵심 요약
*   **목적:** 파이썬 코드 내에서 리눅스 명령어를 실행하고 결과를 제어.
*   **메서드:** `subprocess.run()` (Python 3.5+ 권장)
*   **주요 파라미터:**
    *   `args`: 실행할 명령어 리스트 (예: `["ls", "-l"]`)
    *   `capture_output=True`: 실행 결과를 변수에 담음.
    *   `text=True`: 결과를 문자열로 처리.
    *   `check=True`: 명령어 실패 시 예외 발생.
    *   `shell=True`: 쉘을 통해 실행합니다. (보안상 주의 필요)
    *   `cwd` : 명령어를 실행할 작업 디렉토리를 지정합니다.

In [11]:
def extract_base_md(pdf_path):
  """PyMuPDF를 이용해 텍스트와 페이지 마커가 포함된 기본 .md 파일을 생성합니다."""
  doc = fitz.open(pdf_path)
  base_md = ""
  for i , page in enumerate(doc):
    text = page.get_text()
    base_md += f"\n\n<!-- page: {i+1} --> \n\n{text}\n"
  doc.close()
  return base_md

## 5. 메인 실행 루프 (HWP->PDF->MD)

In [12]:
import fitz
import io # BytesIO 사용을 위해 추가
from PIL import Image

def pdf_to_images(pdf_path: str, dpi : int = 200) -> list[bytes]:
  doc = fitz.open(pdf_path)
  images = [page.get_pixmap(dpi=dpi).tobytes('png') for page in doc]
  doc.close()
  return images

def show_page(image_byte: bytes):
  img = Image.open(io.BytesIO(image_byte))
  display(img.resize((400, int(400* img.height / img.width))))

In [13]:
SYSTEM_PROMPT = """
너는 문서 페이지에서 '표'만 추출하는 도우미다.
보이는 표를 Markdown table로 변환하라. 설명 없이 JSON으로만 답하라.
출력 형식: {"tables": [{"caption": "제목", "markdown": "|...|"}]}
"""

In [17]:
# Colab에서 실행
!apt-get install -y libreoffice
!pip install pyhwp

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libreoffice is already the newest version (1:7.3.7-0ubuntu0.22.04.10).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 22.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 15.3 MB/s eta 0:00:00
  Created wheel for pyhwp: filename=pyhwp-0.1b15-py3-none-any.whl size=315452 sha256=a4721f2381f4c5eb125aa9c9f8e75f9d2eb9b07d76c7958264b837ca3ae2881a
  Stored in directory: /root/.cache/pip/wheels/8e/13/81/cc88f3dcc6e177769677759fffa4ee79fc8eed460d2a36c0cb
Successfully built pyhwp


In [19]:
import subprocess, os

# 명령어 존재 여부 확인
for cmd in ["hwp5odt", "libreoffice"]:
    r = subprocess.run(["which", cmd], capture_output=True, text=True)
    print(f"{cmd} 위치: {r.stdout.strip() or ' 없음'}")

# 파일 1개로 hwp5odt 직접 테스트
test_file = [f for f in os.listdir(FILE_DIR) if f.lower().endswith('.hwp')][0]
test_path = os.path.join(FILE_DIR, test_file)
print(f"\n테스트 파일: {test_file}")

r1 = subprocess.run(["hwp5odt", test_path], capture_output=True, text=True, cwd=OUTPUT_DIR)
print(f"hwp5odt returncode: {r1.returncode}")
print(f"hwp5odt stdout: {r1.stdout[:300]}")
print(f"hwp5odt stderr: {r1.stderr[:300]}")
print(f"\nOUTPUT_DIR 파일 목록: {os.listdir(OUTPUT_DIR)}")

hwp5odt 위치: /usr/local/bin/hwp5odt
libreoffice 위치: /usr/bin/libreoffice

테스트 파일: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp
hwp5odt returncode: 1
hwp5odt stdout: 
hwp5odt stderr: <string>:29:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute font-family-generic for element font-face
<string>:52:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute font-family-generic for element font-face
<string>:72:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute 

OUTPUT_DIR 파일 목록: ['(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.odt']


In [26]:
from qwen_vl_utils import process_vision_info
import os
import re
import json

# 1. HWP -> PDF 변환 실행
print("--- 1. HWP -> PDF 변환 시작 ---")
convert_hwp_to_pdf(FILE_DIR, OUTPUT_DIR)

# 2. 변환된 PDF 파일 확인 및 선택
pdf_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pdf')]

if pdf_files:
    target_pdf = pdf_files[0]
    pdf_path = os.path.join(OUTPUT_DIR, target_pdf)
    print(f"\n[선택된 파일] {pdf_path}")

    # 3. PDF -> 이미지 변환
    images_bytes = pdf_to_images(pdf_path)
    print(f"총 {len(images_bytes)} 페이지 이미지 생성됨")

    # 첫 페이지 미리보기
    show_page(images_bytes[0])

    # 4. AI 모델(Qwen)을 사용하여 표 추출 테스트 (JSON 출력)
    print("\n--- 2. AI 이미지 파싱 (표 추출 - JSON) 시작 ---")

    pil_image = Image.open(io.BytesIO(images_bytes[0]))

    # 사용자가 정의한 시스템 프롬프트 (표 추출 전용)
    SYSTEM_PROMPT = """
   너는 문서 페이지에서 '표'만 추출하는 도우미다.

   규칙:
   - 보이는 표만 추출한다 (추측 금지)
   - 표는 Markdown table로 변환한다
   - 표가 없으면 {"tables": []} 만 반환한다
   - 설명 문장은 쓰지 않는다
   - JSON만 출력한다

   출력 형식:
   {
     "tables": [
       {
         "caption": "표 제목 (없으면 빈 문자열)",
         "markdown": "| ... |"
       }
     ]
   }
   """

    # 메시지 구성
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": "이 페이지에서 표를 찾아 JSON으로 추출해줘."}
            ],
        }
    ]

    # 입력 데이터 전처리
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # 추론
    generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # 결과 처리 및 저장
    print("\n--- [변환 결과 (Raw Text)] ---")
    print(output_text[:500] + "... (생략)" if len(output_text) > 500 else output_text)

    json_filename = target_pdf.replace('.pdf', '_tables.json')
    json_path = os.path.join(OUTPUT_DIR, json_filename)

    try:
        # 안전하게 파싱하여 저장
        parsed_json = safe_json_loads(output_text)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(parsed_json, f, indent=2, ensure_ascii=False)
        print(f"\n[성공] JSON 파싱 및 저장 완료: {json_path}")
        print("\n[추출된 데이터 미리보기]")
        print(json.dumps(parsed_json, indent=2, ensure_ascii=False)[:500])

    except Exception as e:
        print(f"\n[주의] JSON 파싱 실패 ({e}). 원본 텍스트로 저장합니다.")
        with open(json_path, "w", encoding="utf-8") as f:
            f.write(output_text)
        print(f"[저장 완료(Raw)] {json_path}")

else:
    print("\n[주의] 변환할 PDF 파일이 없습니다. FILE_DIR 경로에 .hwp 파일이 있는지 확인해주세요.")

--- 1. HWP -> PDF 변환 시작 ---
발견된 HWP 파일 갯수 : 3
변환 중 : (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp ... 
  [hwp5odt] returncode=1
  [hwp5odt] stderr: <string>:29:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute font-family-generic for element font-face
<string>:52:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute font-family-ge
  [libreoffice] returncode=0
  [libreoffice] stderr: Error: source file could not be loaded

ODT→PDF 실패 (pdf 파일 없음)
변환 중 : (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp ... 
  [hwp5odt] returncode=1
  [hwp5odt] stderr: <string>:131:0:ERROR:RELAXNGV:RELAXNG_ERR_INVALIDATTR: Invalid attribute font-family-generic for element font-face
<string>:409:0:ERROR:RELAXNGV:RELAXNG_ERR_ELEMNAME: Expecting element map, got text-p
  [libreoffice] returncode=0
  [libreoffice] stderr: Error: source file could not be loaded

ODT→PDF 실패 (pdf 파일 없음)
변환 중 : (사)벤처기업협회_2024년 벤ᄎ